# Notebook 03 — Cross-Lingual CKA: Core Similarity Analysis

**TIN-7: Cross-Lingual Embedding Alignment Analysis**

This is the central analysis notebook. It computes pairwise CKA
between all 13 language pairs at each of Tiny Aya's 4 layers,
identifies the convergence layer, and produces the key visualizations.

## What this notebook does

1. Loads cached activations from Notebook 02 (or extracts them).
2. Computes 13x13 Linear CKA matrices at each layer.
3. Computes the average cross-lingual CKA convergence curve.
4. Identifies the convergence layer (where CKA exceeds 0.75).
5. Overlays RBF CKA for comparison.
6. Runs permutation tests for statistical significance.
7. Produces: heatmaps, convergence curves, spaghetti plots.

In [ ]:
import logging
import sys
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from uth.utils.languages import Language
from uth.data.flores_loader import load_flores_parallel_corpus
from uth.analysis.hooks import load_model
from uth.analysis.cross_lingual_alignment import CrossLingualAlignmentAnalyzer
from uth.analysis.cka import linear_cka, rbf_cka, cka_permutation_test
from uth.analysis.visualization import (
    plot_cka_heatmap,
    plot_multi_layer_heatmaps,
    plot_convergence_curve,
    plot_language_pair_trajectories,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

RESULTS_DIR = PROJECT_ROOT / "results" / "cross_lingual"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# --- Configuration ---
PRECISION = "fp16"
BATCH_SIZE = 8
MAX_SENTENCES = None  # None = full FLORES devtest (1012)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Load Model and Data (or Use Cached Activations)

If activations from Notebook 02 are available on disk, we load them
directly. Otherwise, we extract fresh activations.

In [ ]:
model, tokenizer = load_model(precision=PRECISION)
corpus = load_flores_parallel_corpus(max_sentences=MAX_SENTENCES)

analyzer = CrossLingualAlignmentAnalyzer(
    model=model,
    tokenizer=tokenizer,
    parallel_corpus=corpus,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

# Try loading cached activations first.
activation_dir = RESULTS_DIR / "activations"
if activation_dir.exists() and any(activation_dir.iterdir()):
    print("Loading cached activations from disk...")
    analyzer.load_activations(str(RESULTS_DIR))
else:
    print("No cached activations found. Extracting from model...")
    analyzer.extract_all_activations()
    analyzer.save_activations(str(RESULTS_DIR))

## 2. Compute Linear CKA Matrices

For each layer, compute pairwise CKA between all 13x13 language
pairs. This produces the core data for all subsequent analysis.

In [ ]:
cka_matrices = analyzer.compute_cka_matrices(kernel="linear")

# Print summary statistics per layer.
print("\n=== Linear CKA Summary ===")
for layer_idx, matrix in sorted(cka_matrices.items()):
    n = len(analyzer.language_names)
    off_diag = matrix[np.triu_indices(n, k=1)]
    print(
        f"  Layer {layer_idx}: mean={off_diag.mean():.4f}, "
        f"std={off_diag.std():.4f}, "
        f"min={off_diag.min():.4f}, max={off_diag.max():.4f}"
    )

## 3. CKA Heatmaps (One Per Layer)

These heatmaps are the primary deliverable. Each cell shows the CKA
similarity between a pair of languages at that layer. Look for:
- **Diagonal**: Always 1.0 (self-similarity).
- **Off-diagonal convergence**: Values approaching 1.0 in later
  layers indicate language-agnostic representations.

In [ ]:
# Plot individual heatmaps.
iso_names = [lang.iso_code for lang in analyzer.languages]

for layer_idx, matrix in sorted(cka_matrices.items()):
    fig = plot_cka_heatmap(
        cka_matrix=matrix,
        language_names=iso_names,
        layer_index=layer_idx,
        save_path=str(FIGURES_DIR / f"cka_heatmap_layer_{layer_idx}.png"),
    )
    plt.show()

### Combined Multi-Layer Heatmap View

In [ ]:
fig = plot_multi_layer_heatmaps(
    cka_matrices=cka_matrices,
    language_names=iso_names,
    save_path=str(FIGURES_DIR / "cka_heatmaps_all_layers.png"),
)
plt.show()

## 4. Convergence Curve

The convergence curve plots **average cross-lingual CKA** (mean of
all off-diagonal entries) vs. layer depth. The convergence layer is
where this curve exceeds the 0.75 threshold established in the
reference repo's CKA analysis.

In [ ]:
# Compute convergence curve data.
curve = analyzer.compute_convergence_curve()

# Find convergence layer.
convergence_layer = analyzer.find_convergence_layer(threshold=0.75)

print(f"\n=== Convergence Analysis ===")
for i, layer_idx in enumerate(curve["layer_indices"]):
    marker = " <-- CONVERGENCE" if layer_idx == convergence_layer else ""
    print(
        f"  Layer {layer_idx}: avg_CKA={curve['avg_cka'][i]:.4f} "
        f"[{curve['ci_lower'][i]:.4f}, {curve['ci_upper'][i]:.4f}]{marker}"
    )

if convergence_layer is None:
    # Find the best layer even if threshold not reached.
    best_idx = int(np.argmax(curve["avg_cka"]))
    print(f"\nThreshold 0.75 not reached. Best layer: {curve['layer_indices'][best_idx]} "
          f"(avg CKA = {curve['avg_cka'][best_idx]:.4f})")

### Plot Convergence Curve

In [ ]:
fig = plot_convergence_curve(
    layer_indices=curve["layer_indices"],
    avg_cka_per_layer=curve["avg_cka"],
    ci_lower=curve["ci_lower"],
    ci_upper=curve["ci_upper"],
    threshold=0.75,
    save_path=str(FIGURES_DIR / "convergence_curve.png"),
)
plt.show()

## 5. RBF CKA Comparison

RBF CKA captures nonlinear relationships that linear CKA might miss.
We overlay both on the convergence curve.

In [ ]:
rbf_matrices = analyzer.compute_cka_matrices(kernel="rbf")

rbf_curve = analyzer.compute_convergence_curve(cka_matrices=rbf_matrices)

fig = plot_convergence_curve(
    layer_indices=curve["layer_indices"],
    avg_cka_per_layer=curve["avg_cka"],
    ci_lower=curve["ci_lower"],
    ci_upper=curve["ci_upper"],
    rbf_cka_per_layer=rbf_curve["avg_cka"],
    threshold=0.75,
    title="Cross-Lingual CKA Convergence: Linear vs. RBF",
    save_path=str(FIGURES_DIR / "convergence_curve_linear_vs_rbf.png"),
)
plt.show()

## 6. Per-Pair CKA Trajectories (Spaghetti Plot)

Shows how CKA evolves for each individual language pair across layers.
Highlights specific pairs of interest.

In [ ]:
# Compute per-pair CKA trajectories.
trajectories: dict[str, list[float]] = {}

for i in range(len(analyzer.language_names)):
    for j in range(i + 1, len(analyzer.language_names)):
        lang_i = analyzer.languages[i].iso_code
        lang_j = analyzer.languages[j].iso_code
        pair_name = f"{lang_i}-{lang_j}"

        values = []
        for layer_idx in analyzer.layer_indices:
            values.append(cka_matrices[layer_idx][i, j])
        trajectories[pair_name] = values

# Highlight interesting pairs: same family (en-es), different family (en-hi),
# different script (en-ar), low-resource (en-yo).
highlight = ["en-es", "en-hi", "en-ar", "en-yo", "en-sw", "hi-bn"]

fig = plot_language_pair_trajectories(
    layer_indices=analyzer.layer_indices,
    cka_trajectories=trajectories,
    highlight_pairs=highlight,
    save_path=str(FIGURES_DIR / "cka_trajectories_per_pair.png"),
)
plt.show()

## 7. Statistical Significance (Permutation Tests)

Verify that the observed CKA scores are significantly above chance
using permutation tests on a sample of language pairs.

In [ ]:
# Run permutation tests on a few representative pairs at the best layer.
best_layer_idx = (
    convergence_layer if convergence_layer is not None
    else int(np.argmax(curve["avg_cka"]))
)

test_pairs = [
    ("english", "hindi"),
    ("english", "arabic"),
    ("english", "swahili"),
    ("hindi", "bengali"),
    ("french", "spanish"),
]

print(f"\n=== Permutation Tests at Layer {best_layer_idx} ===")
print(f"{'Pair':<20} {'Observed CKA':>12} {'p-value':>10} {'Significant':>12}")
print("-" * 60)

for lang_a_name, lang_b_name in test_pairs:
    layer_name = f"layer_{best_layer_idx}"
    X = analyzer.activations[lang_a_name][layer_name]
    Y = analyzer.activations[lang_b_name][layer_name]

    result = cka_permutation_test(X, Y, n_permutations=500, seed=42)

    iso_a = next(l.iso_code for l in Language if l.lang_name == lang_a_name)
    iso_b = next(l.iso_code for l in Language if l.lang_name == lang_b_name)

    print(
        f"  {iso_a}-{iso_b:<14} {result['observed_cka']:>12.4f} "
        f"{result['p_value']:>10.4f} "
        f"{'YES' if result['is_significant'] else 'NO':>12}"
    )

## 8. Save Metrics

Save all computed metrics as JSON for reproducibility and
downstream analysis.

In [ ]:
# Save convergence curve.
convergence_metrics = {
    f"layer_{layer_idx}_avg_crosslingual_cka": avg_cka
    for layer_idx, avg_cka in zip(curve["layer_indices"], curve["avg_cka"])
}
convergence_metrics["convergence_layer"] = convergence_layer

with open(METRICS_DIR / "convergence_curve.json", "w") as f:
    json.dump(convergence_metrics, f, indent=2)

# Save CKA matrices.
cka_path = RESULTS_DIR / "cka_matrices"
cka_path.mkdir(parents=True, exist_ok=True)
for layer_idx, matrix in cka_matrices.items():
    np.save(cka_path / f"layer_{layer_idx}_linear_cka.npy", matrix)

print(f"\nMetrics saved to {METRICS_DIR}")
print(f"CKA matrices saved to {cka_path}")

## 9. Summary

**Key findings:**
- Cross-lingual CKA matrices computed for all 13x13 pairs x 4 layers.
- Convergence curve identifies the layer of maximum cross-lingual
  alignment.
- Permutation tests confirm statistical significance.
- Next: Notebooks 04-07 for novel analysis techniques.

In [ ]:
print("\n=== Cross-Lingual CKA Analysis Complete ===")
print(f"  Convergence layer: {convergence_layer}")
print(f"  Best avg CKA: {max(curve['avg_cka']):.4f}")
print(f"  Results saved to: {RESULTS_DIR}")